# CIFAR-10 Gaussian: preprocesamiento robusto con XGBoost fijo

## Objetivo

Comparar el baseline de **Hu Moments** contra una representación robusta compuesta por **Wavelet Scattering, HOG, LBP e histogramas HSV**. El clasificador XGBoost y sus parámetros permanecen exactamente iguales al notebook entregado.

## Protocolo

- Entrenamiento: 10 000 imágenes limpias del dataset adjunto.
- Evaluación: las mismas 3 000 imágenes limpias y Gaussian noise con severidades 1–5, reportadas como Nivel 1–5.
- Semilla: `42`.
- Promedio: accuracy limpia más las cinco accuracies corruptas, dividido entre seis.


## 1. Preparar dependencias

Instalamos únicamente `kymatio`, biblioteca no garantizada por Kaggle. CIFAR-10-C se carga con TensorFlow Datasets ya incluido, evitando alterar NumPy o pandas. Salida esperada: instalación silenciosa sin errores.

In [ ]:
%pip install -q --no-deps kymatio

## 2. Importar y fijar configuración

Centralizamos semilla, tamaños, ubicación del Kaggle Dataset y rutas de salida. `run_summary.json` registra el progreso de la ejecución remota. Salida esperada: dataset localizado y dispositivo `cuda` cuando Kaggle asigna T4.


In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as torch_functional
from joblib import Parallel, delayed
from IPython.display import display
from kymatio.torch import Scattering2D
from skimage import color, exposure, filters, morphology
from skimage.feature import hog, local_binary_pattern
from skimage.measure import moments_hu
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

SEED = 42
N_TRAIN = 10_000
N_TEST = 3_000
if not torch.cuda.is_available():
    raise RuntimeError("Este notebook requiere la GPU Nvidia Tesla T4 solicitada")
GPU_NAME = torch.cuda.get_device_name(0)
if "T4" not in GPU_NAME.upper():
    raise RuntimeError(f"Acelerador incorrecto: {GPU_NAME}; se requiere Tesla T4")
DEVICE = "cuda"
KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd() / "outputs-local"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

progress = {"status": "running", "stage": "setup", "seed": SEED, "artifacts": []}


def save_progress():
    (OUTPUT_DIR / "run_summary.json").write_text(
        json.dumps(progress, indent=2, ensure_ascii=False), encoding="utf-8"
    )


save_progress()
print("Dispositivo:", DEVICE, GPU_NAME)
print("Outputs:", OUTPUT_DIR)

## 3. Cargar el Kaggle Dataset

Leemos los `.npy` publicados sin descargarlos desde Internet durante el run. Conservamos exactamente 10 000 ejemplos de entrenamiento y 3 000 de evaluación; las corrupciones nunca participan en `fit`. Salida esperada: shapes `(10000, 32, 32, 3)` y `(3000, 32, 32, 3)`.


In [ ]:
expected_root = KAGGLE_INPUT / "cifar10-clean-gaussian-noise-s1-s5"
if expected_root.exists():
    DATASET_ROOT = expected_root
else:
    candidates = list(KAGGLE_INPUT.rglob("x_train.npy"))
    if len(candidates) != 1:
        raise FileNotFoundError(f"No se encontró un único x_train.npy en {KAGGLE_INPUT}: {candidates}")
    DATASET_ROOT = candidates[0].parent

X_train = np.load(DATASET_ROOT / "x_train.npy", mmap_mode="r")[:N_TRAIN]
y_train = np.load(DATASET_ROOT / "y_train.npy", mmap_mode="r").reshape(-1)[:N_TRAIN]
X_test_clean = np.load(DATASET_ROOT / "x_test_clean.npy", mmap_mode="r")[:N_TEST]
y_test_clean = np.load(DATASET_ROOT / "y_test_clean.npy", mmap_mode="r").reshape(-1)[:N_TEST]

assert X_train.shape == (N_TRAIN, 32, 32, 3)
assert y_train.shape == (N_TRAIN,)
assert X_test_clean.shape == (N_TEST, 32, 32, 3)
assert y_test_clean.shape == (N_TEST,)

progress["stage"] = "clean_data_loaded"
progress["dataset_root"] = str(DATASET_ROOT)
save_progress()
print("Dataset:", DATASET_ROOT)
print("Train:", X_train.shape, y_train.shape)
print("Test limpio:", X_test_clean.shape, y_test_clean.shape)


## 4. Fijar Gaussian noise como Nivel 1–5

La consigna adaptada usa una única corrupción: `gaussian_noise`. Su severidad 1 corresponde a Nivel 1 y así sucesivamente hasta severidad 5. No hay selección aleatoria en `v2`, por lo que la tabla es totalmente reproducible.


In [ ]:
selected_pairs = [("gaussian_noise", severity) for severity in range(1, 6)]
pair_table = pd.DataFrame(
    [
        {"nivel": severity, "corrupcion": corruption, "severidad": severity}
        for corruption, severity in selected_pairs
    ]
)

assert pair_table["nivel"].tolist() == [1, 2, 3, 4, 5]
assert pair_table["severidad"].tolist() == [1, 2, 3, 4, 5]
display(pair_table)


## 5. Cargar las cinco severidades Gaussian

Cada nivel usa las mismas 3 000 observaciones y sus etiquetas entregadas. Verificamos shapes y que las etiquetas coincidan con el test limpio, garantizando una comparación pareada.


In [ ]:
corrupt_tests = []
for level in range(1, 6):
    images = np.load(DATASET_ROOT / f"x_test_gaussian_noise_s{level}.npy", mmap_mode="r")[:N_TEST]
    labels = np.load(DATASET_ROOT / f"y_test_gaussian_noise_s{level}.npy", mmap_mode="r").reshape(-1)[:N_TEST]
    assert images.shape == X_test_clean.shape
    assert labels.shape == y_test_clean.shape
    assert np.array_equal(labels, y_test_clean)
    corrupt_tests.append(
        {
            "nivel": level,
            "corrupcion": "gaussian_noise",
            "severidad": level,
            "images": images,
            "labels": labels,
        }
    )
    print(f"Nivel {level}: gaussian_noise, severidad {level}, {images.shape}, {images.dtype}")

progress["stage"] = "corrupt_data_loaded"
progress["selected_pairs"] = pair_table.to_dict(orient="records")
save_progress()


## 6. Definir baseline Hu Moments

Este transformador reproduce la extracción original: escala a `[0,1]`, convierte a gris y calcula siete Hu Moments. Su propósito es producir una comparación justa con el mismo XGBoost.

In [ ]:
class HuFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []
        for image in X:
            gray = color.rgb2gray(image.astype(np.float32) / 255.0)
            features.append(moments_hu(gray))
        return np.asarray(features, dtype=np.float32)

## 7. Definir features complementarias

- **HOG + CLAHE:** conserva bordes tras cambios de iluminación.
- **LBP + mediana:** resume textura reduciendo ruido impulsivo.
- **HSV espacial:** conserva color y ubicación aproximada.

Usamos una pirámide `1×1 + 2×2` para no perder toda la información espacial.

In [ ]:
def spatial_regions(channel):
    height, width = channel.shape
    return [
        channel,
        channel[: height // 2, : width // 2],
        channel[: height // 2, width // 2 :],
        channel[height // 2 :, : width // 2],
        channel[height // 2 :, width // 2 :],
    ]


def normalized_histogram(values, bins, value_range):
    histogram, _ = np.histogram(values, bins=bins, range=value_range)
    histogram = histogram.astype(np.float32)
    return histogram / max(histogram.sum(), 1.0)


def handcrafted_features(image):
    rgb = image.astype(np.float32) / 255.0
    luminance = color.rgb2gray(rgb)

    clahe = exposure.equalize_adapthist(luminance, clip_limit=0.02, nbins=256)
    hog_vector = hog(
        clahe,
        orientations=9,
        pixels_per_cell=(4, 4),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
    ).astype(np.float32)

    median = filters.median(luminance, footprint=morphology.disk(1))
    lbp = local_binary_pattern((median * 255).astype(np.uint8), P=8, R=1, method="uniform")
    lbp_vector = np.concatenate(
        [normalized_histogram(region, bins=10, value_range=(0, 10)) for region in spatial_regions(lbp)]
    )

    hsv = color.rgb2hsv(rgb)
    hsv_ranges = [(0, 1), (0, 1), (0, 1)]
    hsv_vector = np.concatenate(
        [
            normalized_histogram(region, bins=16, value_range=hsv_ranges[channel])
            for channel in range(3)
            for region in spatial_regions(hsv[:, :, channel])
        ]
    )
    return np.concatenate([hog_vector, lbp_vector, hsv_vector]).astype(np.float32)

## 8. Crear transformador robusto sklearn

Scattering 2D de segundo orden aporta estabilidad frente a pequeñas deformaciones y ruido. Se calcula por lotes en GPU y se reduce a una grilla `4×4`; luego se concatena con HOG, LBP y HSV. Salida: matriz tabular para XGBoost.

In [ ]:
class RobustFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, J=2, max_order=2, batch_size=1024, device=DEVICE):
        self.J = J
        self.max_order = max_order
        self.batch_size = batch_size
        self.device = device

    def fit(self, X, y=None):
        self.device_ = self.device if self.device == "cuda" and torch.cuda.is_available() else "cpu"
        self.scattering_ = Scattering2D(
            J=self.J,
            shape=(32, 32),
            max_order=self.max_order,
        ).to(self.device_)
        return self

    def transform(self, X):
        if not hasattr(self, "scattering_"):
            self.fit(X)

        rgb = np.asarray(X, dtype=np.float32) / 255.0
        luminance = np.tensordot(rgb, np.array([0.2125, 0.7154, 0.0721], dtype=np.float32), axes=([-1], [0]))

        scattering_batches = []
        with torch.no_grad():
            for start in range(0, len(luminance), self.batch_size):
                batch = torch.from_numpy(luminance[start : start + self.batch_size]).to(self.device_)
                coefficients = self.scattering_(batch)
                pooled = torch_functional.adaptive_avg_pool2d(coefficients, output_size=(4, 4))
                scattering_batches.append(pooled.flatten(start_dim=1).cpu().numpy().astype(np.float32))
                print(f"Scattering: {min(start + self.batch_size, len(luminance))}/{len(luminance)}", flush=True)

        scattering_matrix = np.concatenate(scattering_batches, axis=0)
        print(f"HOG/LBP/HSV paralelo: {len(X)} imágenes", flush=True)
        handcrafted_matrix = np.asarray(
            Parallel(n_jobs=-1, prefer="threads")(
                delayed(handcrafted_features)(image) for image in X
            ),
            dtype=np.float32,
        )
        print("HOG/LBP/HSV: completo", flush=True)
        features = np.concatenate([scattering_matrix, handcrafted_matrix], axis=1)
        assert np.isfinite(features).all()
        return features

## 9. Construir pipelines sin modificar XGBoost

Única diferencia entre experimentos: paso `features`. `StandardScaler` y el bloque XGBoost conservan configuración original: objetivo, número de clases, métrica, CPU y semilla.

In [ ]:
CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]


def build_pipeline(feature_extractor):
    return Pipeline(
        [
            ("features", feature_extractor),
            ("scaler", StandardScaler()),
            (
                "model",
                XGBClassifier(
                    objective="multi:softmax",
                    num_class=len(CLASS_NAMES),
                    eval_metric="merror",
                    n_jobs=-1,
                    random_state=42,
                ),
            ),
        ]
    )


baseline_pipeline = build_pipeline(HuFeatureExtractor())
robust_pipeline = build_pipeline(RobustFeatureExtractor())

EXPECTED_XGB_PARAMS = {
    "objective": "multi:softmax",
    "num_class": len(CLASS_NAMES),
    "eval_metric": "merror",
    "n_jobs": -1,
    "random_state": 42,
}
for pipeline in (baseline_pipeline, robust_pipeline):
    model_params = pipeline.named_steps["model"].get_params()
    assert all(model_params[key] == value for key, value in EXPECTED_XGB_PARAMS.items())

print("XGBoost verificado sin cambios:", EXPECTED_XGB_PARAMS)

## 10. Ejecutar smoke test

Antes del cálculo completo, entrenamos clones con 500 imágenes y predecimos 50. Esto detecta incompatibilidades de shapes, GPU o sklearn sin gastar todo el tiempo.

In [ ]:
for name, pipeline in {
    "hu_baseline": baseline_pipeline,
    "scattering_hog_lbp_hsv": robust_pipeline,
}.items():
    smoke_pipeline = clone(pipeline)
    smoke_pipeline.fit(X_train[:500], y_train[:500])
    smoke_predictions = smoke_pipeline.predict(X_test_clean[:50])
    assert smoke_predictions.shape == (50,)
    print(name, "OK", "accuracy=", round(accuracy_score(y_test_clean[:50], smoke_predictions), 4))

progress["stage"] = "smoke_test_complete"
save_progress()

## 11. Entrenar ambos pipelines

Entrenamos Hu y representación robusta sobre exactamente las mismas 10 000 imágenes y etiquetas. CIFAR-10-C sigue fuera del entrenamiento.

In [ ]:
pipelines = {
    "Hu baseline": baseline_pipeline,
    "Scattering + HOG + LBP + HSV": robust_pipeline,
}

for name, pipeline in pipelines.items():
    print("Entrenando:", name)
    pipeline.fit(X_train, y_train)

progress["stage"] = "training_complete"
save_progress()

## 12. Evaluar limpio y los cinco niveles Gaussian

Aplicamos cada pipeline ya entrenado a los mismos seis conjuntos: clean y severidades 1–5. Registramos una fila por estrategia y el detalle auditable de cada nivel.


In [ ]:
metric_rows = []
pair_rows = []

for pipeline_name, pipeline in pipelines.items():
    clean_accuracy = accuracy_score(y_test_clean, pipeline.predict(X_test_clean))
    row = {"pipeline": pipeline_name, "accuracy_test_limpio": clean_accuracy}
    corrupt_accuracies = []

    for test_data in corrupt_tests:
        accuracy = accuracy_score(
            test_data["labels"],
            pipeline.predict(test_data["images"]),
        )
        column = f"accuracy_nivel_{test_data['nivel']}"
        row[column] = accuracy
        corrupt_accuracies.append(accuracy)
        pair_rows.append(
            {
                "pipeline": pipeline_name,
                "nivel": test_data["nivel"],
                "corrupcion": test_data["corrupcion"],
                "severidad": test_data["severidad"],
                "accuracy": accuracy,
            }
        )

    row["promedio"] = float(np.mean([clean_accuracy, *corrupt_accuracies]))
    metric_rows.append(row)

metrics = pd.DataFrame(metric_rows)
selected_pair_metrics = pd.DataFrame(pair_rows)
display(metrics.style.format({column: "{:.4f}" for column in metrics.columns if column != "pipeline"}))

## 13. Validar protocolo y promedio

Comprobamos que hay exactamente cinco niveles Gaussian por pipeline, seis accuracies en `[0,1]` y que el promedio coincide con la media aritmética solicitada por el docente.


In [ ]:
accuracy_columns = ["accuracy_test_limpio"] + [f"accuracy_nivel_{level}" for level in range(1, 6)]

assert metrics.shape[0] == 2
assert selected_pair_metrics.groupby("pipeline").size().eq(5).all()
assert metrics[accuracy_columns].apply(lambda column: column.between(0, 1).all()).all()
assert np.allclose(metrics["promedio"], metrics[accuracy_columns].mean(axis=1))

best_index = metrics["promedio"].idxmax()
print("Mejor pipeline:", metrics.loc[best_index, "pipeline"])
print("Promedio:", f"{metrics.loc[best_index, 'promedio']:.4f}")

## 14. Graficar comparación

La figura resume clean, Gaussian severidad 1–5 y promedio. Queda lista para registrar o sustentar los resultados.


In [ ]:
plot_columns = accuracy_columns + ["promedio"]
plot_labels = ["Clean", "Nivel 1", "Nivel 2", "Nivel 3", "Nivel 4", "Nivel 5", "Promedio"]

ax = metrics.set_index("pipeline")[plot_columns].T.plot(kind="bar", figsize=(13, 5), ylim=(0, 1))
ax.set_xticklabels(plot_labels, rotation=0)
ax.set_ylabel("Accuracy")
ax.set_title("CIFAR-10 limpio + Gaussian noise severidades 1–5")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
comparison_path = OUTPUT_DIR / "comparison.png"
plt.savefig(comparison_path, dpi=160, bbox_inches="tight")
plt.show()

## 15. Guardar resultados finales

Exportamos la tabla general, el detalle Nivel–severidad y el resumen del run. Los archivos se descargarán con Kaggle CLI en `v2/outputs`.


In [ ]:
metrics_path = OUTPUT_DIR / "metrics.csv"
pairs_path = OUTPUT_DIR / "selected_pairs.csv"
metrics.to_csv(metrics_path, index=False)
selected_pair_metrics.to_csv(pairs_path, index=False)

progress.update(
    {
        "status": "complete",
        "stage": "done",
        "best_pipeline": metrics.loc[best_index, "pipeline"],
        "best_average": float(metrics.loc[best_index, "promedio"]),
        "artifacts": [metrics_path.name, pairs_path.name, comparison_path.name, "run_summary.json"],
    }
)
save_progress()

print("Archivos generados:")
for artifact in progress["artifacts"]:
    path = OUTPUT_DIR / artifact
    print("-", path, path.stat().st_size, "bytes")

## 16. Lectura para la defensa

Use los valores ejecutados de la tabla y figura. Explique que XGBoost permaneció fijo; la única diferencia entre filas es el preprocesamiento y la representación. Los niveles 1–5 corresponden directamente a Gaussian noise severidad 1–5.
